<a href="https://colab.research.google.com/github/sadumina/Deep-Learning-Assignment-Group-ID-5/blob/model%2FClassic-CNN/TrainNewCnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
from google.colab import drive
drive.mount('/content/drive')

# find the exact zip path inside your Drive
!find /content/drive/MyDrive -iname "gaussian_filtered_images.zip" 2>/dev/null

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/data/gaussian_filtered_images.zip


In [9]:
!unzip -q /content/gaussian_filtered_images.zip -d /content/dataset

In [10]:
import os
for root, dirs, files in os.walk('/content/dataset'):
    print(root, '->', len(files), 'files,', dirs)
    if root.count('/') > 4:
        break

/content/dataset -> 0 files, ['gaussian_filtered_images']
/content/dataset/gaussian_filtered_images -> 1 files, ['Moderate', 'Severe', 'No_DR', 'Proliferate_DR', 'Mild']
/content/dataset/gaussian_filtered_images/Moderate -> 999 files, []
/content/dataset/gaussian_filtered_images/Severe -> 193 files, []
/content/dataset/gaussian_filtered_images/No_DR -> 1805 files, []
/content/dataset/gaussian_filtered_images/Proliferate_DR -> 295 files, []
/content/dataset/gaussian_filtered_images/Mild -> 370 files, []


In [11]:
import os
root_files = [f for f in os.listdir('/content/dataset/gaussian_filtered_images')
              if os.path.isfile(os.path.join('/content/dataset/gaussian_filtered_images', f))]
print(root_files)

['export.pkl']


In [12]:
DATA_DIR = '/content/dataset/gaussian_filtered_images'
class_names = sorted(['Mild', 'Moderate', 'No_DR', 'Proliferate_DR', 'Severe'])
print(class_names)

['Mild', 'Moderate', 'No_DR', 'Proliferate_DR', 'Severe']


In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split

filepaths, labels = [], []
for cls in class_names:
    cls_path = os.path.join(DATA_DIR, cls)
    for fname in os.listdir(cls_path):
        filepaths.append(os.path.join(cls_path, fname))
        labels.append(cls)

df = pd.DataFrame({'filepath': filepaths, 'label': labels})
print(df['label'].value_counts())

train_df, val_df = train_test_split(
    df, test_size=0.15, stratify=df['label'], random_state=42
)
print("\nTrain distribution:\n", train_df['label'].value_counts())
print("\nVal distribution:\n", val_df['label'].value_counts())

label
No_DR             1805
Moderate           999
Mild               370
Proliferate_DR     295
Severe             193
Name: count, dtype: int64

Train distribution:
 label
No_DR             1534
Moderate           849
Mild               314
Proliferate_DR     251
Severe             164
Name: count, dtype: int64

Val distribution:
 label
No_DR             271
Moderate          150
Mild               56
Proliferate_DR     44
Severe             29
Name: count, dtype: int64


In [14]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

label_to_index = {name: i for i, name in enumerate(class_names)}
train_labels_idx = train_df['label'].map(label_to_index).values

class_weights = compute_class_weight('balanced',
                                      classes=np.unique(train_labels_idx),
                                      y=train_labels_idx)
class_weight_dict = dict(enumerate(class_weights))
for i, cls in enumerate(class_names):
    print(f"{cls}: weight = {class_weight_dict[i]:.2f}")

Mild: weight = 1.98
Moderate: weight = 0.73
No_DR: weight = 0.41
Proliferate_DR: weight = 2.48
Severe: weight = 3.80


In [15]:
import os, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

DATA_DIR = '/content/dataset/gaussian_filtered_images'
class_names = sorted(['Mild', 'Moderate', 'No_DR', 'Proliferate_DR', 'Severe'])

filepaths, labels = [], []
for cls in class_names:
    cls_path = os.path.join(DATA_DIR, cls)
    for fname in os.listdir(cls_path):
        filepaths.append(os.path.join(cls_path, fname))
        labels.append(cls)

df = pd.DataFrame({'filepath': filepaths, 'label': labels})
train_df, val_df = train_test_split(df, test_size=0.15, stratify=df['label'], random_state=42)

label_to_index = {name: i for i, name in enumerate(class_names)}
train_labels_idx = train_df['label'].map(label_to_index).values
class_weights = compute_class_weight('balanced', classes=np.unique(train_labels_idx), y=train_labels_idx)
class_weight_dict = dict(enumerate(class_weights))
print(class_weight_dict)

{0: np.float64(1.9821656050955414), 1: np.float64(0.7330977620730271), 2: np.float64(0.4057366362451108), 3: np.float64(2.4796812749003982), 4: np.float64(3.795121951219512)}


In [28]:
import tensorflow as tf
from tensorflow.keras import layers

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
num_classes = len(class_names)

def load_image(filepath, label):
    img = tf.io.read_file(filepath)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0     # <-- must happen here, on the base image
    return img, label

def make_dataset(df, training):
    paths = df['filepath'].values
    labels = df['label'].map(label_to_index).values
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.shuffle(1000)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df, training=False)

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
    layers.RandomBrightness(0.1),
    layers.RandomContrast(0.1),
])

def augment_and_clip(image, label):
    image = data_augmentation(image, training=True)
    image = tf.clip_by_value(image, 0.0, 1.0)   # <-- force back into valid range
    return image, label

train_ds = train_ds.map(augment_and_clip, num_parallel_calls=tf.data.AUTOTUNE)

In [25]:
for imgs, labels in train_ds.take(1):
    print("min/max pixel values:", imgs.numpy().min(), imgs.numpy().max())

min/max pixel values: 0.0 1.0


In [26]:
from tensorflow.keras import layers, regularizers, Sequential

model = Sequential([
    layers.Input(shape=(224, 224, 3)),

    # Block 1
    layers.Conv2D(32, 3, padding='same', kernel_regularizer=regularizers.l2(1e-4)),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D(),
    layers.Dropout(0.25),

    # Block 2
    layers.Conv2D(64, 3, padding='same', kernel_regularizer=regularizers.l2(1e-4)),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D(),
    layers.Dropout(0.25),

    # Block 3
    layers.Conv2D(128, 3, padding='same', kernel_regularizer=regularizers.l2(1e-4)),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D(),
    layers.Dropout(0.3),

    # Block 4
    layers.Conv2D(256, 3, padding='same', kernel_regularizer=regularizers.l2(1e-4)),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.GlobalAveragePooling2D(),   # replaces Flatten — far fewer params, less overfitting

    # Classifier head
    layers.Dense(128, kernel_regularizer=regularizers.l2(1e-4)),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.5),

    layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_4 (Conv2D)               │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_5 (Activation)       │ (None, 224, 224, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_6 (Activation)       │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_7 (Activation)       │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 28, 28, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 28, 28, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_8 (Activation)       │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_9 (Activation)       │ (None, 128)            │             

 Total params: 424,389 (1.62 MB)

 Trainable params: 423,173 (1.61 MB)

 Non-trainable params: 1,216 (4.75 KB)

In [18]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

drive.mount('/content/drive')  # if not already mounted
os.makedirs('/content/drive/MyDrive/dr_project/checkpoints', exist_ok=True)

callbacks = [
    ModelCheckpoint('/content/drive/MyDrive/dr_project/checkpoints/best_model.keras',
                     monitor='val_loss', save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [29]:
history = model.fit(
    train_ds, validation_data=val_ds, epochs=1,
    class_weight=class_weight_dict, callbacks=callbacks
)

97/98 ━━━━━━━━━━━━━━━━━━━━ 0s 404ms/step - accuracy: 0.2017 - loss: 2.0412
Epoch 1: val_loss did not improve from 1.56241
98/98 ━━━━━━━━━━━━━━━━━━━━ 43s 414ms/step - accuracy: 0.2095 - loss: 1.9745 - val_accuracy: 0.2727 - val_loss: 1.6369 - learning_rate: 2.5000e-05
Restoring model weights from the end of the best epoch: 1.


In [30]:
for imgs, labels in train_ds.take(1):
    print("min/max:", imgs.numpy().min(), imgs.numpy().max())

min/max: 0.0 1.0
